# 03 Inference and Final Submission
This notebook executes the final ranking step within the 5-minute compute budget using cached artifacts. It runs completely offline.

In [21]:
%pip install pandas numpy sentence-transformers pyarrow fastparquet python-docx

Note: you may need to restart the kernel to use updated packages.


In [22]:
import os
import time
import json
import re
import numpy as np
import pandas as pd
import subprocess
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

base_dir = "/content/ai-candidate-ranking"
if not os.path.exists(base_dir):
    base_dir = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
    
raw_dir = os.path.join(base_dir, "data", "raw")
processed_dir = os.path.join(base_dir, "data", "processed")
outputs_dir = os.path.join(base_dir, "outputs", "submissions")
os.makedirs(outputs_dir, exist_ok=True)

### 1. Load Processed Artifacts

In [23]:
print("Loading precomputed artifacts...")
start_time = time.time()

ids_path = os.path.join(processed_dir, "candidate_ids.npy")
candidate_ids = np.load(ids_path, allow_pickle=True)

emb_path = os.path.join(processed_dir, "embeddings.npy")
cand_embeddings = np.load(emb_path)

feat_path = os.path.join(processed_dir, "candidates_feather.parquet")
features_df = pd.read_parquet(feat_path)

print(f"Loaded {len(candidate_ids)} candidates.")
print(f"Loading time: {time.time() - start_time:.2f} seconds")

Loading precomputed artifacts...
Loaded 100000 candidates.
Loading time: 0.16 seconds


### 2. Build and Encode JD Text

In [24]:
def build_jd_text(docx_path):
    try:
        doc = Document(docx_path)
        return "\n".join([p.text.strip() for p in doc.paragraphs if p.text.strip()])
    except Exception as e:
        print(f"Error reading JD: {e}")
        return ""

jd_path = os.path.join(raw_dir, "job_description.docx")
jd_text = build_jd_text(jd_path)

# Load model avoiding network downloads by relying on local cache
model_name = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading {model_name}...")
model = SentenceTransformer(model_name)

print("Encoding JD text...")
jd_embedding = model.encode([jd_text]) if jd_text else np.array([])

Loading sentence-transformers/all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding JD text...


### 3. Compute Semantic Similarity

In [25]:
print("Computing cosine similarities...")
sim_start = time.time()

if cand_embeddings.shape[0] > 0 and jd_embedding.shape[0] > 0:
    similarities = cosine_similarity(jd_embedding, cand_embeddings)[0]
else:
    similarities = np.zeros(len(candidate_ids))

print(f"Similarity computation took: {time.time() - sim_start:.3f} seconds")

sim_df = pd.DataFrame({"candidate_id": candidate_ids, "semantic_similarity": similarities})
full_df = pd.merge(features_df, sim_df, on="candidate_id", how="inner")

Computing cosine similarities...
Similarity computation took: 0.086 seconds


### 4. Final Scoring & Reasoning Generation

In [26]:
weights = {
    "semantic": 0.35, "role": 0.10, "product": 0.10, "exp_band": 0.10,
    "ml_years": 0.10, "ai_depth": 0.05, "avail": 0.05, "rel": 0.05,
    "github": 0.05, "geo": 0.03, "work_mode": 0.02
}
alpha = 0.5
beta = 1.0

max_ml = full_df["ml_years_estimate"].max() if "ml_years_estimate" in full_df.columns else 1.0
if max_ml == 0: max_ml = 1.0

def compute_final_score(row):
    score = (
        row.get("semantic_similarity", 0) * weights["semantic"] +
        row.get("role_title_score", 0) * weights["role"] +
        row.get("product_company_score", 0) * weights["product"] +
        row.get("experience_band_match", 0) * weights["exp_band"] +
        (row.get("ml_years_estimate", 0) / max_ml) * weights["ml_years"] +
        min(row.get("ai_skill_depth_score", 0), 1.0) * weights["ai_depth"] +
        row.get("availability_score", 0) * weights["avail"] +
        row.get("reliability_score", 0) * weights["rel"] +
        row.get("github_fit_score", 0) * weights["github"] +
        row.get("geo_fit_score", 0) * weights["geo"] +
        row.get("work_mode_fit", 0) * weights["work_mode"]
    )
    score -= (row.get("notice_period_penalty", 0) * alpha)
    score -= (row.get("honeypot_risk_score", 0) * beta)
    return max(0.0, min(1.0, score))

full_df["final_score"] = full_df.apply(compute_final_score, axis=1)

# Filter honeypots and sort
filtered_df = full_df[full_df["honeypot_risk_score"] < 0.6].copy()
ranked_df = filtered_df.sort_values(by="final_score", ascending=False).head(100).copy()

print("Extracting raw details for top candidates to generate reasoning...")
top_100_ids = set(ranked_df["candidate_id"].values)
cand_details = {}

cands_path = os.path.join(raw_dir, "candidates.jsonl")
if not os.path.exists(cands_path):
    cands_path = os.path.join(raw_dir, "sample_candidates.json")

try:
    with open(cands_path, 'r') as f:
        first_char = f.read(1)
    if first_char == '[':
        with open(cands_path, "r") as f:
            data = json.load(f)
            for cand in data:
                cid = cand.get("candidate_id")
                if cid in top_100_ids: cand_details[cid] = cand
    else:
        with open(cands_path, "r") as f:
            for line in f:
                if not line.strip(): continue
                cand = json.loads(line)
                cid = cand.get("candidate_id")
                if cid in top_100_ids: cand_details[cid] = cand
except Exception as e:
    print(f"Could not load raw data: {e}")

ai_titles = ["machine learning", "deep learning", "nlp", "search", "retrieval", 
             "embedding", "vector", "pytorch", "tensorflow", "mlflow", "llm", "ai"]
ai_pattern = re.compile("|".join(ai_titles), re.IGNORECASE)

def generate_reasoning(row):
    cid = row["candidate_id"]
    title = row.get("current_title", "Professional")
    yoe = row.get("total_years_experience", 0)
    ai_count = row.get("ai_core_skills_count", 0)
    sim = row.get("semantic_similarity", 0)
    
    named_skills = []
    signals = {}
    if cid in cand_details:
        cand = cand_details[cid]
        skills = cand.get("skills", [])
        signals = cand.get("redrob_signals", {})
        skill_names = [s.get("name", str(s)) if isinstance(s, dict) else str(s) for s in skills]
        named_skills = [s for s in skill_names if ai_pattern.search(s)][:3]
        
    if sim > 0.6: alignment = "exceptional alignment with JD requirements for semantic ranking"
    elif sim > 0.4: alignment = "strong background in machine learning and embeddings"
    else: alignment = "moderate exposure to AI tooling"
        
    try: resp_rate = float(signals.get("recruiter_response_rate", 0))
    except: resp_rate = 0.0
    last_act = signals.get("last_active_date", "")
    
    if resp_rate > 0.8: behavior = f"Highly responsive to recruiters ({int(resp_rate*100)}% rate)"
    elif last_act: behavior = f"Recently active on the platform ({last_act[:10]})"
    else: behavior = "Moderate platform engagement"
        
    sentence1 = f"An experienced {title} with {yoe} years of experience, demonstrating {alignment}."
    skills_str = f" Includes core AI competencies like {', '.join(named_skills)} among {ai_count} ML skills." if named_skills else f" Displays {ai_count} relevant AI skills."
    sentence2 = f" {behavior}."
    
    concern = ""
    if row.get("product_company_score", 0) < 0.3: concern = " However, background leans heavily towards IT services."
    elif row.get("notice_period_penalty", 0) > 0: concern = " Note: Long notice period."
    elif row.get("ml_years_estimate", 0) < 1.0 and ai_count > 5: concern = " Note: Many AI skills but limited explicit ML tenure."
        
    return sentence1 + skills_str + sentence2 + concern

ranked_df["reasoning"] = ranked_df.apply(generate_reasoning, axis=1)

Extracting raw details for top candidates to generate reasoning...


### 5. Format Submission

In [27]:
ranked_df["score"] = ranked_df["final_score"].round(3)
ranked_df["rank"] = range(1, len(ranked_df) + 1)

# Enforce monotonically non-increasing scores
prev_score = ranked_df.iloc[0]["score"]
adjusted_scores = []
for s in ranked_df["score"]:
    if s > prev_score:
        adjusted_scores.append(prev_score)
    else:
        adjusted_scores.append(s)
        prev_score = s
ranked_df["score"] = adjusted_scores

submission_df = ranked_df[["candidate_id", "rank", "score", "reasoning"]].copy()

sub_path = os.path.join(outputs_dir, "ranked_candidates.csv")
submission_df.to_csv(sub_path, index=False)
print(f"Saved final submission to {sub_path}")

Saved final submission to ../outputs/submissions/ranked_candidates.csv


### 6. Validation and Inspection

In [28]:
val_script = os.path.join(base_dir, "validate_submission.py")
if not os.path.exists(val_script):
    val_script = os.path.join(raw_dir, "validate_submission.py")

meta_path = os.path.join(base_dir, "submission_metadata.yaml")
if not os.path.exists(meta_path):
    meta_path = os.path.join(raw_dir, "submission_metadata.yaml")
    
print("=== Running Validator ===")
import sys
try:
    result = subprocess.run([sys.executable, val_script, sub_path, meta_path], capture_output=True, text=True)
    print("STDOUT:\n" + result.stdout)
    if result.stderr:
        print("STDERR:\n" + result.stderr)
except Exception as e:
    print(f"Failed to run validator: {e}")

print("\n=== Top 10 rows ===")
display(submission_df.head(10))

print("\n=== 5 Random Sample rows ===")
display(submission_df.sample(min(5, len(submission_df))))

=== Running Validator ===
STDOUT:
Validating ../outputs/submissions/ranked_candidates.csv...
Error: Missing required columns. Expected at least: {'candidate_id', 'job_id', 'score'}


=== Top 10 rows ===


,candidate_id,rank,score,reasoning
0,CAND_0000001,1,1.0,An experienced Professional with 6.9 years of ...
56966,CAND_0056967,2,1.0,An experienced Professional with 11.1 years of...
22167,CAND_0022168,3,1.0,An experienced Professional with 2.8 years of ...
56949,CAND_0056950,4,1.0,An experienced Professional with 3.3 years of ...
56951,CAND_0056952,5,1.0,An experienced Professional with 6.4 years of ...
56953,CAND_0056954,6,1.0,An experienced Professional with 12.9 years of...
56956,CAND_0056957,7,1.0,An experienced Professional with 9.7 years of ...
22162,CAND_0022163,8,1.0,An experienced Professional with 9.2 years of ...
22161,CAND_0022162,9,1.0,An experienced Professional with 1.8 years of ...
56958,CAND_0056959,10,1.0,An experienced Professional with 1.6 years of ...



=== 5 Random Sample rows ===


,candidate_id,rank,score,reasoning
56893,CAND_0056894,39,1.0,An experienced Professional with 6.1 years of ...
56967,CAND_0056968,16,1.0,An experienced Professional with 9.6 years of ...
22089,CAND_0022090,78,1.0,An experienced Professional with 3.7 years of ...
56874,CAND_0056875,35,1.0,An experienced Professional with 8.8 years of ...
56927,CAND_0056928,55,1.0,An experienced Professional with 11.7 years of...
